# NYC Shooting Incidents: Precinct Map Explorer

This notebook creates one map-based Plotly visualization for the combined NYPD shooting data. It uses `data/nypd_shootings_combined.csv` together with `final_project/output/nyc_police_precincts.geojson`, then writes the interactive output to `final_project/output/nyc_shootings_precinct_map_explorer.html`.

The map colors precincts by borough and deepens each borough color according to the selected shooting concentration. Controls let the reader adjust years, borough, fatality status, demographic filters, metric, and whether individual shooting dots are shown.


In [ ]:

from pathlib import Path
import json

import pandas as pd
from plotly.offline import get_plotlyjs


if Path("data/nypd_shootings_combined.csv").exists():
    data_path = Path("data/nypd_shootings_combined.csv")
    geojson_path = Path("final_project/output/nyc_police_precincts.geojson")
    output_dir = Path("final_project/output")
elif Path("../data/nypd_shootings_combined.csv").exists():
    data_path = Path("../data/nypd_shootings_combined.csv")
    geojson_path = Path("output/nyc_police_precincts.geojson")
    output_dir = Path("output")
else:
    raise FileNotFoundError("Could not find data/nypd_shootings_combined.csv")

if not geojson_path.exists():
    raise FileNotFoundError("Could not find nyc_police_precincts.geojson")

output_dir.mkdir(parents=True, exist_ok=True)
html_path = output_dir / "nyc_shootings_precinct_map_explorer.html"


df = pd.read_csv(data_path, parse_dates=["datetime"])
df = df.dropna(subset=["year", "BORO", "PRECINCT", "Latitude", "Longitude"]).copy()
df["year"] = df["year"].astype(int)
df["month"] = df["month"].fillna(0).astype(int)
df["hour"] = df["hour"].fillna(0).astype(int)
df["PRECINCT"] = df["PRECINCT"].astype(int).astype(str)
df["num_victims"] = df["num_victims"].fillna(0).astype(int)
df["num_murdered"] = df["num_murdered"].fillna(0).astype(int)
df["has_fatality"] = df["num_murdered"] > 0

for col in [
    "BORO",
    "day_of_week",
    "victim_age_groups",
    "victim_sexes",
    "victim_races",
    "perp_age_groups",
    "perp_sexes",
    "perp_races",
    "LOC_CLASSFCTN_DESC",
]:
    df[col] = df[col].fillna("Unknown").astype(str)

geojson = json.loads(geojson_path.read_text(encoding="utf-8"))
for feature in geojson["features"]:
    feature["id"] = str(feature["properties"]["precinct"])

precinct_boro = (
    df.groupby("PRECINCT")["BORO"]
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0])
    .to_dict()
)
precincts = sorted([str(feature["properties"]["precinct"]) for feature in geojson["features"]], key=lambda x: int(x))

records = df[[
    "INCIDENT_KEY", "year", "month", "day_of_week", "hour", "BORO", "PRECINCT",
    "Latitude", "Longitude", "num_victims", "num_murdered", "has_fatality",
    "victim_age_groups", "victim_sexes", "victim_races", "perp_age_groups",
    "perp_sexes", "perp_races", "LOC_CLASSFCTN_DESC",
]].rename(columns={
    "INCIDENT_KEY": "id",
    "BORO": "boro",
    "PRECINCT": "precinct",
    "Latitude": "lat",
    "Longitude": "lon",
    "LOC_CLASSFCTN_DESC": "location_class",
}).to_dict(orient="records")

meta = {
    "minYear": int(df["year"].min()),
    "maxYear": int(df["year"].max()),
    "boroughs": sorted(df["BORO"].unique().tolist()),
    "precincts": precincts,
    "precinctBoro": {p: precinct_boro.get(p, "UNKNOWN") for p in precincts},
    "totals": {
        "incidents": int(len(df)),
        "victims": int(df["num_victims"].sum()),
        "fatalities": int(df["num_murdered"].sum()),
    },
}

plotly_js = get_plotlyjs()
records_json = json.dumps(records, separators=(",", ":"))
geojson_json = json.dumps(geojson, separators=(",", ":"))
meta_json = json.dumps(meta, separators=(",", ":"))

html_template = """<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>NYC Shooting Precinct Map Explorer</title>
  <script>__PLOTLY_JS__</script>
  <style>
    :root { --bg: #f7f7f4; --panel: #ffffff; --ink: #202124; --muted: #5d646b; --line: #d9ddd6; --active: #202124; }
    * { box-sizing: border-box; }
    body { margin: 0; font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, \"Segoe UI\", sans-serif; color: var(--ink); background: var(--bg); }
    header { padding: 20px 28px 12px; border-bottom: 1px solid var(--line); background: #fbfbf8; }
    h1 { margin: 0 0 7px; font-size: 27px; line-height: 1.15; font-weight: 760; letter-spacing: 0; }
    .subtitle { margin: 0; max-width: 1100px; color: var(--muted); font-size: 14px; line-height: 1.45; }
    .viz-layout { display: grid; grid-template-columns: minmax(0, 1fr) 390px; gap: 16px; padding: 14px 28px 24px; align-items: start; }
    .controls { min-height: 760px; display: grid; grid-template-columns: 1fr; gap: 14px; align-content: start; padding: 16px; background: #eff1ec; border: 1px solid var(--line); border-radius: 8px; overflow: auto; }
    .control-group { display: grid; gap: 10px; padding: 12px; background: rgba(255, 255, 255, 0.72); border: 1px solid var(--line); border-radius: 8px; }
    .control-title { color: var(--ink); font-size: 12px; font-weight: 780; text-transform: uppercase; letter-spacing: 0; }
    label { display: grid; gap: 5px; color: #3f464b; font-size: 11px; font-weight: 720; text-transform: uppercase; letter-spacing: 0; }
    select, input { min-width: 0; width: 100%; border: 1px solid #bfc7c0; border-radius: 6px; padding: 8px 9px; background: white; color: var(--ink); font-size: 13px; }
    .button-row { display: flex; gap: 6px; flex-wrap: wrap; }
    button { border: 1px solid #bfc7c0; border-radius: 6px; padding: 8px 10px; background: white; color: var(--ink); font-size: 13px; cursor: pointer; }
    button.active { border-color: var(--active); background: var(--active); color: white; }
    .toggle { display: flex; align-items: center; gap: 8px; min-height: 36px; text-transform: none; font-size: 13px; font-weight: 650; }
    .toggle input { width: auto; }
    .map-wrap { min-width: 0; }
    .kpis { display: grid; grid-template-columns: repeat(4, minmax(130px, 1fr)); gap: 10px; margin-bottom: 10px; }
    .kpi { background: white; border: 1px solid var(--line); border-radius: 8px; padding: 10px 12px; }
    .kpi strong { display: block; font-size: 22px; line-height: 1.1; }
    .kpi span { display: block; margin-top: 4px; color: var(--muted); font-size: 12px; }
    #map { width: 100%; height: 760px; border: 1px solid var(--line); border-radius: 8px; overflow: hidden; background: white; }
    .note { margin: 8px 2px 0; color: var(--muted); font-size: 12px; line-height: 1.4; }
    @media (max-width: 980px) { .viz-layout { grid-template-columns: 1fr; padding: 14px; } .controls { min-height: 0; grid-template-columns: 1fr 1fr; } }
    @media (max-width: 700px) { header { padding-left: 14px; padding-right: 14px; } .controls { grid-template-columns: 1fr; } .kpis { grid-template-columns: 1fr 1fr; } #map { height: 620px; } }
  </style>
</head>
<body>
  <header>
    <h1>NYC Shooting Incidents by Police Precinct</h1>
    <p class=\"subtitle\">Precinct color identifies borough, while deeper color means a higher selected concentration of shootings. Use the sliders and buttons to decide which incidents appear; turn the dots on to inspect individual shootings.</p>
  </header>
  <main class=\"viz-layout\">
  <section class=\"map-wrap\">
    <section class=\"kpis\"><div class=\"kpi\"><strong id=\"kpiIncidents\">0</strong><span>shooting incidents</span></div><div class=\"kpi\"><strong id=\"kpiVictims\">0</strong><span>people shot</span></div><div class=\"kpi\"><strong id=\"kpiFatalities\">0</strong><span>fatalities</span></div><div class=\"kpi\"><strong id=\"kpiPeak\">0</strong><span>highest precinct concentration</span></div></section>
    <div id=\"map\"></div>
    <p class=\"note\">Source: combined NYPD shooting incidents dataset and NYC police precinct GeoJSON. One incident with missing coordinates is excluded from dot mapping.</p>
  </section>
  <section class=\"controls\" aria-label=\"Map controls\">
    <div class=\"control-group\">
    <div class=\"control-title\">Time</div>
    <label><span>Start year: <span id=\"startYearLabel\">__MIN_YEAR__</span></span><input id=\"startYear\" type=\"range\" min=\"__MIN_YEAR__\" max=\"__MAX_YEAR__\" value=\"__MIN_YEAR__\"></label>
    <label><span>End year: <span id=\"endYearLabel\">__MAX_YEAR__</span></span><input id=\"endYear\" type=\"range\" min=\"__MIN_YEAR__\" max=\"__MAX_YEAR__\" value=\"__MAX_YEAR__\"></label>
    <label>Month<select id=\"monthFilter\"><option value=\"All\">All months</option></select></label>
    <label>Weekday<select id=\"dayFilter\"><option value=\"All\">All days</option></select></label>
    <label><span>Start hour: <span id=\"hourStartLabel\">00:00</span></span><input id=\"hourStart\" type=\"range\" min=\"0\" max=\"23\" value=\"0\"></label>
    <label><span>End hour: <span id=\"hourEndLabel\">23:00</span></span><input id=\"hourEnd\" type=\"range\" min=\"0\" max=\"23\" value=\"23\"></label>
    </div>
    <div class=\"control-group\">
    <div class=\"control-title\">Map metric</div>
    <label>Metric<select id=\"metric\"><option value=\"incidents\">Incidents</option><option value=\"victims\">People shot</option><option value=\"fatalities\">Fatalities</option><option value=\"fatalRate\">Fatal incident rate</option></select></label>
    </div>
    <div class=\"control-group\">
    <div class=\"control-title\">Place and incident type</div>
    <label>Borough<select id=\"borough\"><option value=\"All\">All boroughs</option></select></label>
    <label>Fatality<span class=\"button-row\"><button type=\"button\" class=\"active\" data-fatal=\"All\">All</button><button type=\"button\" data-fatal=\"Fatal\">Fatal</button><button type=\"button\" data-fatal=\"Nonfatal\">Nonfatal</button></span></label>
    </div>
    <div class=\"control-group\">
    <div class=\"control-title\">People involved</div>
    <label>Victim age<select id=\"victimAge\"><option value=\"All\">All victim ages</option></select></label>
    <label>Perpetrator age<select id=\"perpAge\"><option value=\"All\">All perpetrator ages</option></select></label>
    <label>Demographic<select id=\"demoField\"><option value=\"none\">No demographic filter</option><option value=\"victim_races\">Victim race</option><option value=\"victim_age_groups\">Victim age</option><option value=\"victim_sexes\">Victim sex</option><option value=\"perp_races\">Perpetrator race</option><option value=\"perp_age_groups\">Perpetrator age</option><option value=\"perp_sexes\">Perpetrator sex</option></select></label>
    <label>Value<select id=\"demoValue\"><option value=\"All\">All values</option></select></label>
    </div>
    <div class=\"control-group\">
    <div class=\"control-title\">Display</div>
    <label class=\"toggle\"><input id=\"showDots\" type=\"checkbox\" checked>Show shooting locations</label>
    </div>
  </section>
  </main>
  <script>
    const records = __RECORDS__;
    const geojson = __GEOJSON__;
    const meta = __META__;
    const boroughColors = { "BROOKLYN": {low: "#cfe8e4", high: "#0f766e"}, "BRONX": {low: "#f0cbd0", high: "#b23a48"}, "QUEENS": {low: "#cdddec", high: "#335c81"}, "MANHATTAN": {low: "#eadfbd", high: "#7a5c1c"}, "STATEN ISLAND": {low: "#ded4ea", high: "#6a4c93"}, "UNKNOWN": {low: "#eeeeee", high: "#777777"} };
    const boroughOrder = ["BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND"];
    let fatalMode = "All";
    let activeRangeInput = null;
    const boroughSelect = document.getElementById("borough");
    boroughOrder.forEach(boro => { const opt = document.createElement("option"); opt.value = boro; opt.textContent = boro; boroughSelect.appendChild(opt); });
    const monthNames = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"];
    const dayOrder = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"];
    function addOptions(selectId, values) { const select = document.getElementById(selectId); values.forEach(item => { const opt = document.createElement("option"); opt.value = item.value; opt.textContent = item.label; select.appendChild(opt); }); }
    function sortedValues(field) { const counts = new Map(); records.forEach(d => splitValues(d[field]).forEach(v => { const normalized = v.toUpperCase() === "UNKNOWN" ? "Unknown" : v; counts.set(normalized, (counts.get(normalized) || 0) + 1); })); return Array.from(counts.keys()).filter(v => v !== "Unknown").sort((a, b) => { const order = ["<18", "18-24", "25-44", "45-64", "65+"]; return (order.indexOf(a) === -1 ? 99 : order.indexOf(a)) - (order.indexOf(b) === -1 ? 99 : order.indexOf(b)) || a.localeCompare(b); }).map(v => ({value: v, label: v})); }
    addOptions("monthFilter", monthNames.map((name, idx) => ({value: String(idx + 1), label: name})));
    addOptions("dayFilter", dayOrder.map(day => ({value: day, label: day})));
    addOptions("victimAge", sortedValues("victim_age_groups"));
    addOptions("perpAge", sortedValues("perp_age_groups"));
    function fmt(x) { return Number(x).toLocaleString("en-US"); }
    function pct(x) { return `${(Number(x) || 0).toFixed(1)}%`; }
    function splitValues(value) { if (!value || value === "Unknown" || value === "nan") return ["Unknown"]; return String(value).split("|").map(v => v.trim()).filter(Boolean); }
    function syncYearLabels() { const startInput = document.getElementById("startYear"); const endInput = document.getElementById("endYear"); let start = Number(startInput.value); let end = Number(endInput.value); if (end < start) { if (activeRangeInput === "endYear") { end = start; endInput.value = end; } else { start = end; startInput.value = start; } } document.getElementById("startYearLabel").textContent = start; document.getElementById("endYearLabel").textContent = end; return {start, end}; }
    function syncHourLabels() { const startInput = document.getElementById("hourStart"); const endInput = document.getElementById("hourEnd"); let start = Number(startInput.value); let end = Number(endInput.value); if (end < start) { if (activeRangeInput === "hourEnd") { end = start; endInput.value = end; } else { start = end; startInput.value = start; } } document.getElementById("hourStartLabel").textContent = `${String(start).padStart(2, "0")}:00`; document.getElementById("hourEndLabel").textContent = `${String(end).padStart(2, "0")}:00`; return {start, end}; }
    function populateDemoValues() { const field = document.getElementById("demoField").value; const select = document.getElementById("demoValue"); select.innerHTML = '<option value="All">All values</option>'; if (field === "none") return; const counts = new Map(); records.forEach(d => splitValues(d[field]).forEach(v => counts.set(v, (counts.get(v) || 0) + 1))); Array.from(counts.entries()).filter(([v]) => v !== "Unknown").sort((a, b) => b[1] - a[1] || a[0].localeCompare(b[0])).forEach(([value]) => { const opt = document.createElement("option"); opt.value = value; opt.textContent = value; select.appendChild(opt); }); }
    function filteredRecords() { const years = syncYearLabels(); const hours = syncHourLabels(); const boro = document.getElementById("borough").value; const month = document.getElementById("monthFilter").value; const day = document.getElementById("dayFilter").value; const victimAge = document.getElementById("victimAge").value; const perpAge = document.getElementById("perpAge").value; const demoField = document.getElementById("demoField").value; const demoValue = document.getElementById("demoValue").value; return records.filter(d => { if (d.year < years.start || d.year > years.end) return false; if (d.hour < hours.start || d.hour > hours.end) return false; if (month !== "All" && Number(d.month) !== Number(month)) return false; if (day !== "All" && d.day_of_week !== day) return false; if (boro !== "All" && d.boro !== boro) return false; if (fatalMode === "Fatal" && !d.has_fatality) return false; if (fatalMode === "Nonfatal" && d.has_fatality) return false; if (victimAge !== "All" && !splitValues(d.victim_age_groups).includes(victimAge)) return false; if (perpAge !== "All" && !splitValues(d.perp_age_groups).includes(perpAge)) return false; if (demoField !== "none" && demoValue !== "All" && !splitValues(d[demoField]).includes(demoValue)) return false; return true; }); }
    function metricValue(bucket, metric) { if (metric === "victims") return bucket.victims; if (metric === "fatalities") return bucket.fatalities; if (metric === "fatalRate") return bucket.incidents ? 100 * bucket.fatalIncidents / bucket.incidents : 0; return bucket.incidents; }
    function aggregateByPrecinct(data) { const buckets = new Map(); meta.precincts.forEach(p => buckets.set(p, {incidents: 0, victims: 0, fatalities: 0, fatalIncidents: 0})); data.forEach(d => { const p = String(d.precinct); if (!buckets.has(p)) buckets.set(p, {incidents: 0, victims: 0, fatalities: 0, fatalIncidents: 0}); const b = buckets.get(p); b.incidents += 1; b.victims += Number(d.num_victims || 0); b.fatalities += Number(d.num_murdered || 0); b.fatalIncidents += d.has_fatality ? 1 : 0; }); return buckets; }
    function tracesFor(data) { const metric = document.getElementById("metric").value; const selectedBoro = document.getElementById("borough").value; const showDots = document.getElementById("showDots").checked; const buckets = aggregateByPrecinct(data); const values = meta.precincts.map(p => metricValue(buckets.get(p), metric)); const maxValue = Math.max(1, ...values); const traces = []; boroughOrder.forEach(boro => { const locations = meta.precincts.filter(p => meta.precinctBoro[p] === boro); const z = locations.map(p => metricValue(buckets.get(p), metric)); const customdata = locations.map(p => { const b = buckets.get(p); return [p, boro, b.incidents, b.victims, b.fatalities, b.incidents ? 100 * b.fatalIncidents / b.incidents : 0]; }); traces.push({ type: "choroplethmapbox", geojson, locations, z, featureidkey: "properties.precinct", zmin: 0, zmax: maxValue, colorscale: [[0, boroughColors[boro].low], [1, boroughColors[boro].high]], marker: {line: {color: "#ffffff", width: 1.15}, opacity: selectedBoro === "All" || selectedBoro === boro ? 0.82 : 0.18}, showscale: false, name: boro, customdata, hovertemplate: "Precinct %{customdata[0]}<br>%{customdata[1]}<br>Incidents: %{customdata[2]:,}<br>People shot: %{customdata[3]:,}<br>Fatalities: %{customdata[4]:,}<br>Fatal incident rate: %{customdata[5]:.1f}%<extra></extra>" }); }); traces.push({ type: "scattermapbox", lat: data.map(d => d.lat), lon: data.map(d => d.lon), mode: "markers", visible: showDots, marker: { size: data.map(d => d.has_fatality ? 8 : 5), color: data.map(d => d.has_fatality ? "#b23a48" : "#202124"), opacity: 0.68 }, text: data.map(d => `${d.boro} precinct ${d.precinct}`), customdata: data.map(d => [d.year, d.month, d.hour, d.num_victims, d.num_murdered, d.location_class]), name: "Shooting dots", hoverinfo: showDots ? "all" : "skip", hovertemplate: showDots ? "%{text}<br>Year: %{customdata[0]}<br>Month: %{customdata[1]} | Hour: %{customdata[2]}:00<br>People shot: %{customdata[3]}<br>Fatalities: %{customdata[4]}<br>%{customdata[5]}<extra></extra>" : null }); return {traces, maxValue}; }
    function updateKpis(data, maxValue) { const incidents = data.length; const victims = data.reduce((s, d) => s + Number(d.num_victims || 0), 0); const fatalities = data.reduce((s, d) => s + Number(d.num_murdered || 0), 0); const metric = document.getElementById("metric").value; document.getElementById("kpiIncidents").textContent = fmt(incidents); document.getElementById("kpiVictims").textContent = fmt(victims); document.getElementById("kpiFatalities").textContent = fmt(fatalities); document.getElementById("kpiPeak").textContent = metric === "fatalRate" ? pct(maxValue) : fmt(maxValue); }
    function render() { const data = filteredRecords(); const result = tracesFor(data); updateKpis(data, result.maxValue); const metricLabel = document.getElementById("metric").selectedOptions[0].textContent; const layout = { title: {text: `Precinct concentration by ${metricLabel.toLowerCase()}`, x: 0.02, xanchor: "left"}, margin: {l: 0, r: 0, t: 44, b: 0}, paper_bgcolor: "white", font: {family: "Inter, system-ui, sans-serif", color: "#202124"}, showlegend: true, legend: {orientation: "h", x: 0.01, y: 0.01, bgcolor: "rgba(255,255,255,0.82)"}, mapbox: {style: "white-bg", center: {lat: 40.7128, lon: -73.94}, zoom: 9.55}, uirevision: "keep-map-position" }; Plotly.react("map", result.traces, layout, {responsive: true, displaylogo: false}); }
    document.querySelectorAll("button[data-fatal]").forEach(btn => { btn.addEventListener("click", () => { fatalMode = btn.dataset.fatal; document.querySelectorAll("button[data-fatal]").forEach(b => b.classList.toggle("active", b === btn)); render(); }); });
    ["startYear", "endYear", "hourStart", "hourEnd"].forEach(id => { const el = document.getElementById(id); el.addEventListener("input", () => { activeRangeInput = id; render(); }); el.addEventListener("change", () => { activeRangeInput = id; render(); }); });
    ["monthFilter", "dayFilter", "metric", "borough", "victimAge", "perpAge", "demoValue", "showDots"].forEach(id => { document.getElementById(id).addEventListener("input", render); document.getElementById(id).addEventListener("change", render); });
    document.getElementById("demoField").addEventListener("change", () => { populateDemoValues(); render(); });
    populateDemoValues(); render();
  </script>
</body>
</html>
"""

html = (html_template
    .replace('__PLOTLY_JS__', plotly_js)
    .replace('__RECORDS__', records_json)
    .replace('__GEOJSON__', geojson_json)
    .replace('__META__', meta_json)
    .replace('__MIN_YEAR__', str(meta['minYear']))
    .replace('__MAX_YEAR__', str(meta['maxYear']))
)

html_path.write_text(html, encoding='utf-8')
print(f"Wrote {html_path.resolve()}")
print(meta['totals'])
